# HSC-iM xenograft: independent RNA-only reclustering

**Objective:** Recompute RNA-only dimensionality reduction, unsupervised clustering, UMAP, and gene-expression comparisons for published xenograft HSC/HSC-II cells from the original filtered 10x gene-expression counts (PBS/TNF/LPS).

**Not an exact reproduction of the paper's joint RNA+ATAC WNN analysis.** The original cell labels are used only to *select* the HSC/HSC-II population and for validation/marker comparisons. The new clustering and RNA UMAP are calculated independently from gene-expression counts. This analysis does not rerun SoupX, ATAC QC, donor demultiplexing, or joint WNN integration. Cells are not independent biological replicates: no donor-level treatment inference is made.

**Sources:** Zeng, Nagree et al., *Human haematopoietic stem cells remember inflammatory stress*, Nature (2026), DOI 10.1038/s41586-026-10522-7. Published code/data: https://zenodo.org/records/18850871

**Usage:** In Google Colab, choose **Runtime → Run all**. The notebook downloads only the relevant files from the authors' Zenodo ZIP, processing the large count files *one at a time*. Outputs are saved under `/content/hsc_im_independent_rna/` and packaged for download. Requires internet and a Colab session. Approximate runtime depends on Zenodo transfer speed and Colab availability.


In [ ]:
%pip -q install remotezip scanpy igraph leidenalg

from pathlib import Path
from remotezip import RemoteZip
from collections import Counter
import pandas as pd
import numpy as np
import scipy.sparse as sp
import gzip, gc, json, shutil, time
import scanpy as sc
import matplotlib.pyplot as plt
from zipfile import ZipFile

sc.settings.verbosity = 2
np.random.seed(123)
ROOT = Path('/content/hsc_im_independent_rna')
FIGURES = ROOT / 'figures'
TABLES = ROOT / 'results'
for d in [ROOT, FIGURES, TABLES]: d.mkdir(parents=True, exist_ok=True)
URL = 'https://zenodo.org/records/18850871/files/DATASETS%20-%20NOTEBOOKS%200-5.zip?download=1'
PREFIX = 'DATASETS - NOTEBOOKS 0-5/'
CONDITIONS = ['PBS', 'TNF', 'LPS']
print('Environment ready. Scanpy:', sc.__version__)

In [ ]:
# Get published cell IDs and annotations for HSC population selection / later validation.
embedding_path = ROOT / 'published_WNN_embedding.csv'
with RemoteZip(URL) as archive:
    member = PREFIX + 'PBSTNFLPS_JointEmbedding_WNN_byCondition.csv'
    with archive.open(member) as source, open(embedding_path, 'wb') as dest:
        shutil.copyfileobj(source, dest, length=1024 * 1024)

ref = pd.read_csv(embedding_path)
required = {'Cell', 'Treatment', 'Cluster', 'UMAP1_WNN', 'UMAP2_WNN'}
if not required.issubset(ref.columns):
    raise ValueError(f'Unexpected published embedding columns: {ref.columns.tolist()}')
ref = ref[ref['Treatment'].isin(CONDITIONS)].drop_duplicates('Cell').copy()
keep = ref[ref['Cluster'].isin(['HSC', 'HSC-II'])].copy()
if keep.empty: raise ValueError('Could not find HSC or HSC-II annotations.')
keep['Barcode'] = [c[len(t)+1:] if c.startswith(t+'_') else c for c,t in zip(keep.Cell,keep.Treatment)]
print('Published WNN all-cell count:', len(ref))
print('HSC-only selection (used to define subpopulation, NOT cluster assignments):')
print(pd.crosstab(keep.Treatment, keep.Cluster).to_string())
if len(keep) < 1000: raise RuntimeError('Unexpectedly small HSC subset; check labels.')

In [ ]:
# Download and STREAM each full 10x matrix, retaining ONLY RNA counts for the published HSC subset.
# Unlike scipy.io.mmread of the full multiome matrix, this avoids holding giant ATAC matrices in RAM.

def download_member(archive, member, target):
    target = Path(target)
    if target.exists() and target.stat().st_size:
        return target
    print('  Downloading:', member.rsplit('/',1)[-1], flush=True)
    temp = target.with_suffix(target.suffix+'.partial')
    try:
        with archive.open(member) as src, open(temp, 'wb') as dest:
            shutil.copyfileobj(src, dest, length=1024*1024)
        temp.rename(target)
    finally:
        if temp.exists(): temp.unlink()
    return target

def read_lines_gzip(file):
    with gzip.open(file, 'rt') as f:
        return [line.rstrip('\n') for line in f]

def extract_hsc_expression(condition):
    base = f'{PREFIX}inputs/Rawdata/{condition}/filtered_feature_bc_matrix/'
    cache = ROOT / 'temp'; cache.mkdir(exist_ok=True)
    with RemoteZip(URL) as archive:
        fs = download_member(archive, base + 'features.tsv.gz', cache / f'{condition}_features.tsv.gz')
        bs = download_member(archive, base + 'barcodes.tsv.gz', cache / f'{condition}_barcodes.tsv.gz')
        features = pd.read_csv(fs, sep='\t', compression='gzip', header=None, dtype=str)
        if features.shape[1] < 3: raise ValueError(f'{condition}: 10x feature type column missing')
        gene_rows = np.flatnonzero((features.iloc[:,2] == 'Gene Expression').to_numpy())
        if len(gene_rows) < 1000: raise ValueError(f'{condition}: no Gene Expression features')
        barcodes = read_lines_gzip(bs)
        subset = keep[keep.Treatment == condition].set_index('Barcode')
        selected = [i for i,b in enumerate(barcodes) if b in subset.index]
        print(f'{condition}: matched {len(selected)}/{len(subset)} published HSC barcodes; genes={len(gene_rows)}', flush=True)
        if len(selected) < len(subset)*0.7: raise ValueError(f'{condition}: barcode mismatch; stopping to avoid invalid analysis')
        matfile = download_member(archive, base + 'matrix.mtx.gz', cache / f'{condition}_matrix.mtx.gz')

    gene_index = np.full(len(features), -1, dtype=np.int32)
    gene_index[gene_rows] = np.arange(len(gene_rows), dtype=np.int32)
    cell_index = np.full(len(barcodes), -1, dtype=np.int32)
    cell_index[selected] = np.arange(len(selected), dtype=np.int32)
    rows, cols, vals = [], [], []
    scanned = 0
    print(f'{condition}: streaming count entries (not loading entire multiome)...', flush=True)
    with gzip.open(matfile, 'rt') as stream:
        header = next(stream).strip()
        if not header.startswith('%%MatrixMarket matrix coordinate'):
            raise ValueError('Unexpected MatrixMarket format: ' + header)
        shape_line = next(stream)
        while shape_line.startswith('%'): shape_line = next(stream)
        num_features, num_cells, num_entries = map(int, shape_line.split())
        if num_features != len(features) or num_cells != len(barcodes):
            raise ValueError(f'{condition}: mismatched feature/barcode dimensions')
        for line in stream:
            fields = line.split()
            if len(fields) != 3: continue
            r, c, v = int(fields[0])-1, int(fields[1])-1, float(fields[2])
            gr = gene_index[r]; cc = cell_index[c]
            if gr >= 0 and cc >= 0:
                rows.append(cc); cols.append(gr); vals.append(v)
            scanned += 1
            if scanned % 8_000_000 == 0:
                print(f'  {condition}: scanned {scanned:,} / ~{num_entries:,} matrix entries', flush=True)
    mat = sp.coo_matrix((np.asarray(vals,dtype=np.float32),
                        (np.asarray(rows,dtype=np.int32),np.asarray(cols,dtype=np.int32))),
                       shape=(len(selected), len(gene_rows))).tocsr()
    picked_barcodes = [barcodes[i] for i in selected]
    matching = subset.loc[picked_barcodes]
    adata = sc.AnnData(X=mat)
    adata.var_names = pd.Index(features.iloc[gene_rows,0].astype(str).to_numpy())
    adata.var['gene_symbol'] = features.iloc[gene_rows,1].astype(str).to_numpy()
    adata.obs_names = matching['Cell'].astype(str).to_numpy()
    adata.obs['treatment'] = condition
    adata.obs['published_type'] = matching['Cluster'].astype(str).to_numpy()
    adata.obs['published_umap1'] = matching['UMAP1_WNN'].to_numpy()
    adata.obs['published_umap2'] = matching['UMAP2_WNN'].to_numpy()
    print(f'{condition}: retained {adata.n_obs:,} HSC cells x {adata.n_vars:,} RNA genes, {mat.nnz:,} nonzeros', flush=True)
    matfile.unlink(missing_ok=True)
    del mat, rows, cols, vals, features, barcodes
    gc.collect()
    return adata

chunks = []
for condition in CONDITIONS:
    chunks.append(extract_hsc_expression(condition))

# Retain intersection of gene IDs when building combined RNA expression matrix.
adata = sc.concat(chunks, join='inner', merge='first', index_unique=None)
adata.var_names_make_unique()
del chunks
gc.collect()
print('Combined HSC RNA matrix:', adata.shape, 'nonzeros:', adata.X.nnz)
print('Counts by published type:')
print(pd.crosstab(adata.obs.treatment, adata.obs.published_type).to_string())
adata.write_h5ad(ROOT/'hsc_rna_selected_raw_counts.h5ad', compression='gzip')

In [ ]:
# Independent RNA-only normalization, PCA, unsupervised clustering and UMAP.
# Labels published_type are not used in the clustering algorithm.
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key='treatment', flavor='seurat')
sc.tl.pca(adata, n_comps=40, use_highly_variable=True, svd_solver='arpack', random_state=123)
sc.pp.neighbors(adata, n_neighbors=20, n_pcs=30, random_state=123)
sc.tl.umap(adata, random_state=123)
sc.tl.leiden(adata, resolution=0.5, key_added='rna_leiden', flavor='igraph', directed=False, n_iterations=2)

fig, axs = plt.subplots(1, 3, figsize=(16,5))
for ax, key, title in zip(axs, ['rna_leiden','published_type','treatment'],
                          ['Independent RNA-only clusters','Published HSC/HSC-II labels','Treatment']):
    sc.pl.umap(adata, color=key, ax=ax, show=False, title=title, size=6, frameon=False)
fig.suptitle('HSC-only xenograft RNA re-clustering (independent RNA UMAP)', y=1.03)
fig.tight_layout()
fig.savefig(FIGURES/'independent_HSC_RNA_umap.png', dpi=250, bbox_inches='tight')
plt.show(); plt.close(fig)

comparison = pd.crosstab(adata.obs['rna_leiden'], adata.obs['published_type'])
comparison.to_csv(TABLES/'rna_clusters_vs_published_HSC_labels.csv')
print("Independent cluster versus published HSC annotation:")
print(comparison.to_string())

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
metrics = {
    'n_cells': int(adata.n_obs),
    'n_genes_after_filtering': int(adata.n_vars),
    'n_independent_clusters': int(adata.obs.rna_leiden.nunique()),
    'ARI_vs_published_binary_HSC_label': float(adjusted_rand_score(adata.obs.published_type,adata.obs.rna_leiden)),
    'NMI_vs_published_binary_HSC_label': float(normalized_mutual_info_score(adata.obs.published_type,adata.obs.rna_leiden)),
    'analysis_scope': 'RNA-only reclustering; not original multiome WNN'
}
(TABLES/'analysis_metrics.json').write_text(json.dumps(metrics,indent=2))
print(metrics)

In [ ]:
# Recalculate RNA marker genes from the original 10x counts (normalized log data).
# Uses the authors' HSC/HSC-II labels for the grouping; this is new DE computation, not independent label discovery.
sc.tl.rank_genes_groups(adata, groupby='published_type', groups=['HSC-II'], reference='HSC', method='wilcoxon', use_raw=False)
markers = sc.get.rank_genes_groups_df(adata, group='HSC-II')
markers['gene_symbol'] = markers['names'].map(adata.var['gene_symbol'].to_dict())
markers.to_csv(TABLES/'HSCII_vs_HSC_RNA_markers_recomputed.csv', index=False)
print('Top newly computed HSC-II/HSC RNA markers:')
print(markers.head(15).to_string(index=False))

# Plotting routines return Axes or lists of Axes in some Scanpy releases.
# Save with Matplotlib rather than assuming a Figure return value.
sc.pl.rank_genes_groups(adata, n_genes=15, sharey=False, show=False)
plt.gcf().savefig(FIGURES/'HSCII_RNA_marker_rankings.png', dpi=250, bbox_inches='tight')
plt.close('all')

# Save small cell-level table for validation and potential figure recreation.
cellinfo = adata.obs[['treatment','published_type','rna_leiden']].copy()
cellinfo['rna_umap1'] = adata.obsm['X_umap'][:,0]
cellinfo['rna_umap2'] = adata.obsm['X_umap'][:,1]
cellinfo.to_csv(TABLES/'independent_RNA_UMAP_cell_annotations.csv', index_label='Cell')

# Provenance and reproducibility notes.
readme = """# Independent RNA-only xenograft HSC re-clustering

Source: Zeng, Nagree et al. (2026), *Human haematopoietic stem cells remember inflammatory stress*, Nature, doi:10.1038/s41586-026-10522-7. Data: https://zenodo.org/records/18850871

This analysis starts from the published 10x **filtered feature-barcode matrices** for PBS/TNF/LPS. The authors' published annotations are used to select HSC/HSC-II cells and compare labels. Gene counts are then independently normalized, highly variable genes selected, PCA/neighbors computed, Leiden clusters identified, RNA UMAP computed, and Wilcoxon-ranked HSC-II versus HSC marker genes calculated.

**Limitations:** This is not the paper's full RNA+ATAC WNN reproduction; it does not repeat SoupX ambient RNA correction, multiome QC, sex/donor demultiplexing, ATAC analysis, Harmony/batch integration, or the published WNN clustering. Cell-type selection and HSC/HSC-II grouping use authors' annotations; treatment comparisons are descriptive only. Differences from the published figure are expected. No donor-level statistical conclusions should be drawn from treating cells as replicates.

Run: open `HSC_iM_Independent_RNA_Reclustering.ipynb` in Google Colab and choose Runtime > Run all. Input is downloaded from Zenodo using HTTP range reads (no need for the whole ZIP locally). Save the generated outputs, as Colab temporary storage is ephemeral.
"""
(ROOT/'README.md').write_text(readme)

# Include a copy of this notebook in the zip by downloading the .ipynb alongside the results.
from google.colab import files
zip_path = Path('/content/HSC_iM_Independent_RNA_Results.zip')
with ZipFile(zip_path, 'w') as z:
    for f in ROOT.rglob('*'):
        if f.is_file() and f.name not in {'hsc_rna_selected_raw_counts.h5ad', 'published_WNN_embedding.csv'} and 'temp' not in f.relative_to(ROOT).parts:
            z.write(f, arcname=str(f.relative_to(ROOT)))
print('Output ZIP:', zip_path, 'size MB:',round(zip_path.stat().st_size/1e6,2))
files.download(str(zip_path))
print('All done. Download and save your ZIP and notebook.')